In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

X_train = pd.read_csv('../data/X_train.csv', index_col='ROW_ID')
y_train = pd.read_csv('../data/y_train.csv', index_col='ROW_ID')
total_data = pd.concat([X_train, y_train], axis=1)

In [5]:
def add_cross_sectional_zscore(X, base_features, suffix='_ZSCORE'):
    """
    For each feature in base_features, compute its z-score relative to the
    cross-section of all allocations present at the same TS.
    z = (x - mean(x at this TS)) / std(x at this TS)

    This isolates relative strength on a given day, independent of that
    day's overall market regime (volatile vs calm days), which raw values
    conflate.
    """
    new_cols = []
    for feat in base_features:
        ts_mean = X.groupby('TS')[feat].transform('mean')
        ts_std = X.groupby('TS')[feat].transform('std')

        new_col = f'{feat}{suffix}'
        # avoid division by zero on days where the feature is constant across the universe
        X[new_col] = np.where(ts_std > 0, (X[feat] - ts_mean) / ts_std, 0.0)
        new_cols.append(new_col)

    return X, new_cols

In [6]:
base_features_to_standardize = ['RET_1', 'MEDIAN_DAILY_TURNOVER']

X_train, new_cols_train = add_cross_sectional_zscore(X_train, base_features_to_standardize)

print(new_cols_train)
print(X_train[new_cols_train].describe())

['RET_1_ZSCORE', 'MEDIAN_DAILY_TURNOVER_ZSCORE']
       RET_1_ZSCORE  MEDIAN_DAILY_TURNOVER_ZSCORE
count  5.270730e+05                  5.234350e+05
mean   1.921031e-19                  2.341621e-19
std    9.976056e-01                  9.975890e-01
min   -9.453999e+00                 -4.323929e-01
25%   -5.853899e-01                 -3.699870e-01
50%    1.251288e-03                 -3.161623e-01
75%    5.882467e-01                 -1.971293e-01
max    1.501519e+01                  6.758235e+00


In [7]:
ts_ret1_std = X_train.groupby('TS')['RET_1'].std()
print(ts_ret1_std.describe())
print((ts_ret1_std < ts_ret1_std.quantile(0.01)).sum(), "days in the bottom 1% of cross-sectional RET_1 spread")

count    2522.000000
mean        0.002860
std         0.000985
min         0.001024
25%         0.002197
50%         0.002652
75%         0.003248
max         0.011833
Name: RET_1, dtype: float64
26 days in the bottom 1% of cross-sectional RET_1 spread
